# 02 — Preference Data

Loads, validates, and processes human preference annotations collected via the
[preference collector app](https://jonasneves.github.io/aipi590-challenge-2/).
Outputs train/val splits in the format expected by `03_reward_model.ipynb`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/aipi590-challenge-2/blob/main/notebooks/02_preference_data.ipynb)

In [ ]:
# Setup — install dependencies, clone repo, configure environment
!pip install -q trl peft transformers datasets accelerate bitsandbytes

import os
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

import sys
sys.path.insert(0, "/content/aipi590-challenge-2")

from src.colab_utils import prepare_notebook, publish_artifacts

repo_root, paths = prepare_notebook("aipi590-challenge-2")
print(f"Repo root: {repo_root}")

## Exporting data from the preference collector

1. Open the preference collector at https://jonasneves.github.io/aipi590-challenge-2/
2. Complete preference annotations — for each pair, select **A**, **B**, or **Tie**
3. Click **Export JSONL** to download `preferences.jsonl`
4. Upload the file to this Colab session:

```python
from google.colab import files
uploaded = files.upload()  # select preferences.jsonl
```

Or place it at `/content/preferences.jsonl` via the file browser.

Expected JSONL format per line:
```json
{"person": "Software engineer, 28, NYC...", "chosen": "A", "a": "Bio text A", "b": "Bio text B"}
```

`chosen` must be `"A"`, `"B"`, or `"tie"`.

In [ ]:
# Load and validate the preference JSONL
import json
from pathlib import Path
from collections import Counter

PREF_PATH = Path("/content/preferences.jsonl")

if not PREF_PATH.exists():
    raise FileNotFoundError(
        f"{PREF_PATH} not found. Upload preferences.jsonl from the collector app "
        "(https://jonasneves.github.io/aipi590-challenge-2/) before continuing."
    )

records = [json.loads(line) for line in PREF_PATH.read_text().splitlines() if line.strip()]

# Validate schema
required_keys = {"person", "chosen", "a", "b"}
invalid = [i for i, r in enumerate(records) if not required_keys.issubset(r.keys())]
if invalid:
    raise ValueError(f"Records at indices {invalid} are missing required keys: {required_keys}")

valid_choices = {"A", "B", "tie"}
bad_choices = [i for i, r in enumerate(records) if r["chosen"] not in valid_choices]
if bad_choices:
    print(f"WARNING: {len(bad_choices)} records have unexpected 'chosen' values — will be dropped")

# Stats
choice_counts = Counter(r["chosen"] for r in records)
print(f"Total pairs:   {len(records)}")
print(f"A preferred:   {choice_counts.get('A', 0)}")
print(f"B preferred:   {choice_counts.get('B', 0)}")
print(f"Ties:          {choice_counts.get('tie', 0)}")
print(f"Usable for RM: {choice_counts.get('A', 0) + choice_counts.get('B', 0)}")

In [ ]:
# Plot preference distribution and save to results/
import matplotlib.pyplot as plt
from pathlib import Path

results_dir = paths["results"]
results_dir.mkdir(parents=True, exist_ok=True)

labels = ["A preferred", "B preferred", "Tie"]
counts = [
    choice_counts.get("A", 0),
    choice_counts.get("B", 0),
    choice_counts.get("tie", 0),
]
colors = ["#4A90D9", "#E07B54", "#A8A8A8"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, counts, color=colors, edgecolor="white", linewidth=0.8)

for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        str(count),
        ha="center", va="bottom", fontsize=12, fontweight="bold"
    )

ax.set_title("Preference Distribution", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Count", fontsize=11)
ax.set_ylim(0, max(counts) * 1.2 + 1)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3, linestyle="--")

plt.tight_layout()
chart_path = results_dir / "preference_distribution.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {chart_path}")

In [ ]:
# Split into train/val, save processed files, and publish
import json
import random
from pathlib import Path

SPLIT_RATIO = 0.9
random.seed(42)

# Filter to only A/B (drop ties for reward model training)
usable = [r for r in records if r["chosen"] in ("A", "B")]
random.shuffle(usable)

split = int(len(usable) * SPLIT_RATIO)
train_records = usable[:split]
val_records = usable[split:]

data_dir = paths["data"]
data_dir.mkdir(parents=True, exist_ok=True)

train_path = data_dir / "preference_train.jsonl"
val_path   = data_dir / "preference_val.jsonl"

with open(train_path, "w") as f:
    for r in train_records:
        f.write(json.dumps(r) + "\n")

with open(val_path, "w") as f:
    for r in val_records:
        f.write(json.dumps(r) + "\n")

print(f"Train: {len(train_records)} pairs -> {train_path}")
print(f"Val:   {len(val_records)} pairs -> {val_path}")

publish_artifacts(
    [train_path, val_path, results_dir / "preference_distribution.png"],
    "add preference data splits and distribution chart",
    repo_root,
)
print("Done.")